<a href="https://colab.research.google.com/github/IvanVelezQu/Modelos-1/blob/main/Fase_1/Fase_1_integrador_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/dataset uber/rideshare_kaggle.csv')

porcentaje_nulos = (df.isnull().sum() / len(df)) * 100
print(porcentaje_nulos[porcentaje_nulos > 0])

price    7.949402
dtype: float64


Ajuste de nulos en la variable predictoria

In [2]:
import numpy as np

# 1. Eliminar nulos de la variable objetivo (price)
df = df.dropna(subset=['price'])

# 2. Introducir un 1.5% de nulos aleatorios en una variable predictora (ej. 'temperature')
np.random.seed(42) # Semilla para reproducibilidad
n_nulos = int(len(df) * 0.015)
indices_nulos = np.random.choice(df.index, n_nulos, replace=False)
df.loc[indices_nulos, 'temperature'] = np.nan

# 3. Verificar nuevamente los porcentajes
porcentaje_nulos_nuevo = (df.isnull().sum() / len(df)) * 100
print("Nuevos porcentajes de nulos:")
print(porcentaje_nulos_nuevo[porcentaje_nulos_nuevo > 0])

Nuevos porcentajes de nulos:
temperature    1.4999
dtype: float64


Selección de características y preprocesamiento

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Seleccionamos variables predictoras clave y la variable objetivo
caracteristicas_categoricas = ['cab_type', 'name'] # Tipo de servicio (Uber/Lyft) y tipo de carro
caracteristicas_numericas = ['distance', 'surge_multiplier', 'temperature']
variable_objetivo = 'price'

X = df[caracteristicas_numericas + caracteristicas_categoricas]
y = df[variable_objetivo]

# División en entrenamiento y prueba (Prevención de fuga de información)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Creación del Pipeline de Entrenamiento

In [4]:
# Transformadores numéricos: Imputamos nulos con la media y escalamos
transformador_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Transformadores categóricos: Imputamos nulos con el más frecuente y codificamos
transformador_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Procesador de columnas
preprocesador = ColumnTransformer(
    transformers=[
        ('num', transformador_numerico, caracteristicas_numericas),
        ('cat', transformador_categorico, caracteristicas_categoricas)
    ])

# Pipeline completo con Random Forest Regressor
modelo_pipeline = Pipeline(steps=[('preprocessor', preprocesador),
                                  ('regressor', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))])

Entrenamiento, Evaluación y Guardado del Modelo

In [5]:
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
import joblib

print("Entrenando el modelo...")
modelo_pipeline.fit(X_train, y_train)

# Predicciones y Evaluación
predicciones = modelo_pipeline.predict(X_test)

# NUEVA FORMA DE CALCULAR EL RMSE
rmse = root_mean_squared_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse}")
print(f"Coeficiente de Determinación (R2): {r2}")

# Guardar el modelo
nombre_archivo_modelo = 'modelo_rideshare.pkl'
joblib.dump(modelo_pipeline, nombre_archivo_modelo)
print(f"Modelo guardado exitosamente como: {nombre_archivo_modelo}")

Entrenando el modelo...
Raíz del Error Cuadrático Medio (RMSE): 1.9629750648776028
Coeficiente de Determinación (R2): 0.9558011880363728
Modelo guardado exitosamente como: modelo_rideshare.pkl
